# 🔥 SafeSense-VI: ViDeBERTa Training on Google Colab

**Vietnamese Toxic Comment Classification with Text Pair**

- Model: `Fsoft-AIC/videberta-base`
- Task: Multi-class (Clean/Toxic/Hate)
- Key Feature: Text Pair tokenization (Title + Comment)
- Expected F1: 0.72-0.78

---

## 📋 Setup Instructions

1. Upload file `final_train_data_v3_SEMANTIC.xlsx` vào Colab
2. Runtime → Change runtime type → T4 GPU
3. Chạy từng cell theo thứ tự

## Cell 1: Install Dependencies

In [ ]:
# Fix pyarrow compatibility
!pip install pyarrow==14.0.2 -q
!pip install transformers datasets accelerate openpyxl -q

import torch
print(f'✅ PyTorch: {torch.__version__}')
print(f'✅ CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
    print(f'✅ Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

## Cell 2: Import Libraries

In [ ]:
import os
import torch
import pandas as pd
import numpy as np
from datasets import Dataset, ClassLabel, Features, Value
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)
from sklearn.metrics import f1_score, accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

print('✅ All libraries imported!')

## Cell 3: Configuration

In [ ]:
class Config:
    MODEL_NAME = "Fsoft-AIC/videberta-base"
    NUM_LABELS = 3
    MAX_LENGTH = 256
    
    BATCH_SIZE = 8
    GRADIENT_ACCUMULATION = 4  # Effective batch = 32
    EPOCHS = 5
    LEARNING_RATE = 2e-5
    WEIGHT_DECAY = 0.01
    WARMUP_RATIO = 0.1
    
    SEED = 42
    OUTPUT_DIR = "./videberta_v2_output"
    
    # Special tokens
    SPECIAL_TOKENS = ["<person>", "<user>", "<emo_pos>", "<emo_neg>"]
    SEP_TOKEN = "<sep>"

# Set seed
torch.manual_seed(Config.SEED)
np.random.seed(Config.SEED)

print("="*60)
print("CONFIGURATION")
print("="*60)
print(f"Model: {Config.MODEL_NAME}")
print(f"Max Length: {Config.MAX_LENGTH}")
print(f"Batch Size: {Config.BATCH_SIZE} x {Config.GRADIENT_ACCUMULATION} = {Config.BATCH_SIZE * Config.GRADIENT_ACCUMULATION}")
print(f"Learning Rate: {Config.LEARNING_RATE}")
print(f"Epochs: {Config.EPOCHS}")

## Cell 4: Upload & Load Data

**⚠️ QUAN TRỌNG:** Upload file `final_train_data_v3_SEMANTIC.xlsx` vào Colab trước khi chạy cell này!

In [ ]:
from google.colab import files

print("📤 Upload file final_train_data_v3_SEMANTIC.xlsx")
uploaded = files.upload()

# Get filename
data_file = list(uploaded.keys())[0]
print(f"\n✅ Uploaded: {data_file}")

In [ ]:
print("="*60)
print("LOADING DATA")
print("="*60)

df = pd.read_excel(data_file)
df.columns = df.columns.str.strip()
df = df.dropna(subset=["training_text", "label"])
df["label"] = df["label"].astype(int)

print(f"✅ Loaded: {len(df)} samples")
print(f"✅ Columns: {list(df.columns)}")
print(f"\n📊 Label distribution:")
print(df['label'].value_counts().sort_index())

# Check data format
sample = df['training_text'].iloc[0]
print(f"\n📝 Sample text:")
print(sample[:150] + "...")
print(f"\n✅ Has <sep>: {'<sep>' in sample}")

## Cell 5: Load Tokenizer & Add Special Tokens

In [ ]:
print("="*60)
print("LOADING TOKENIZER")
print("="*60)

tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME, use_fast=True)
print(f"✅ Tokenizer loaded: {Config.MODEL_NAME}")
print(f"   Vocab size (before): {len(tokenizer)}")

# Add special tokens
num_added = tokenizer.add_tokens(Config.SPECIAL_TOKENS)
print(f"✅ Added {num_added} special tokens: {Config.SPECIAL_TOKENS}")
print(f"   Vocab size (after): {len(tokenizer)}")

## Cell 6: Tokenization Function (Text Pair)

**🔑 KEY FIX:** Split title/comment và dùng `text_pair` để model hiểu cấu trúc

In [ ]:
def tokenize_function(examples):
    """
    Split text into context (title) and comment using <sep>
    Then use text_pair for proper segment handling
    """
    raw_texts = examples["training_text"]
    
    contexts = []
    comments = []
    
    for text in raw_texts:
        text = str(text)
        
        if Config.SEP_TOKEN in text:
            # Split by <sep> token
            parts = text.split(Config.SEP_TOKEN, 1)
            context = parts[0].strip()
            comment = parts[1].strip() if len(parts) > 1 else ""
            
            contexts.append(context if context else "")
            comments.append(comment if comment else "")
        else:
            # No separator - use whole text as context
            contexts.append(text.strip())
            comments.append("")
    
    # Tokenize with text_pair
    return tokenizer(
        text=contexts,
        text_pair=comments,
        truncation=True,
        max_length=Config.MAX_LENGTH,
        padding=False  # DataCollator will handle padding
    )

print("✅ Tokenization function defined (with text_pair)")

## Cell 7: Create Dataset

In [ ]:
print("="*60)
print("CREATING DATASET")
print("="*60)

# Define features with ClassLabel for stratification
features = Features({
    'training_text': Value('string'),
    'label': ClassLabel(num_classes=Config.NUM_LABELS, names=['Clean', 'Toxic', 'Hate'])
})

dataset = Dataset.from_pandas(df[['training_text', 'label']], features=features)

print(f"✅ Dataset created: {len(dataset)} samples")

# Split train/val with stratification
dataset = dataset.train_test_split(test_size=0.15, seed=Config.SEED, stratify_by_column='label')
train_dataset = dataset['train']
val_dataset = dataset['test']

print(f"✅ Train: {len(train_dataset)} samples")
print(f"✅ Val: {len(val_dataset)} samples")

# Tokenize
print("\n🔄 Tokenizing...")
train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=['training_text'])
val_dataset = val_dataset.map(tokenize_function, batched=True, remove_columns=['training_text'])

print(f"✅ Tokenization complete")
print(f"   Train columns: {train_dataset.column_names}")

## Cell 8: Load Model

In [ ]:
print("="*60)
print("LOADING MODEL")
print("="*60)

model = AutoModelForSequenceClassification.from_pretrained(
    Config.MODEL_NAME,
    num_labels=Config.NUM_LABELS,
    id2label={0: "Clean", 1: "Toxic", 2: "Hate"},
    label2id={"Clean": 0, "Toxic": 1, "Hate": 2}
)

# Resize embeddings for new tokens
model.resize_token_embeddings(len(tokenizer))

print(f"✅ Model loaded: {Config.MODEL_NAME}")
print(f"✅ Embeddings resized to: {len(tokenizer)}")
print(f"   Parameters: {sum(p.numel() for p in model.parameters()):,}")

## Cell 9: Metrics Function

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average='macro'),
        "f1_weighted": f1_score(labels, preds, average='weighted')
    }

print("✅ Metrics function defined")

## Cell 10: Training Arguments

In [ ]:
print("="*60)
print("SETTING UP TRAINING")
print("="*60)

training_args = TrainingArguments(
    output_dir=Config.OUTPUT_DIR,
    
    # Training params
    num_train_epochs=Config.EPOCHS,
    per_device_train_batch_size=Config.BATCH_SIZE,
    per_device_eval_batch_size=Config.BATCH_SIZE * 2,
    gradient_accumulation_steps=Config.GRADIENT_ACCUMULATION,
    
    # Optimizer
    learning_rate=Config.LEARNING_RATE,
    weight_decay=Config.WEIGHT_DECAY,
    warmup_ratio=Config.WARMUP_RATIO,
    
    # Evaluation
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    
    # Logging
    logging_steps=50,
    logging_dir=f"{Config.OUTPUT_DIR}/logs",
    
    # Performance
    fp16=True,
    
    # Save
    save_total_limit=2,
    
    # Misc
    seed=Config.SEED,
    report_to="none"
)

# Data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(f"✅ Training arguments set")
print(f"   Epochs: {Config.EPOCHS}")
print(f"   Effective batch: {Config.BATCH_SIZE * Config.GRADIENT_ACCUMULATION}")
print(f"   Learning rate: {Config.LEARNING_RATE}")

## Cell 11: Create Trainer & Train 🏋️

**⏱️ Thời gian dự kiến:** ~15-20 phút (5 epochs)

In [ ]:
print("="*60)
print("🏋️ STARTING TRAINING")
print("="*60)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# Train!
train_result = trainer.train()

print("\n" + "="*60)
print("🏆 TRAINING COMPLETE!")
print("="*60)
print(f"Training loss: {train_result.training_loss:.4f}")

## Cell 12: Final Evaluation

In [ ]:
print("="*60)
print("📊 FINAL EVALUATION")
print("="*60)

eval_results = trainer.evaluate()

print(f"\n📊 Results:")
print(f"   Loss: {eval_results['eval_loss']:.4f}")
print(f"   Accuracy: {eval_results['eval_accuracy']:.4f}")
print(f"   F1 (macro): {eval_results['eval_f1_macro']:.4f}")
print(f"   F1 (weighted): {eval_results['eval_f1_weighted']:.4f}")

# Detailed classification report
predictions = trainer.predict(val_dataset)
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

print(f"\n📋 Classification Report:")
print(classification_report(labels, preds, target_names=['Clean', 'Toxic', 'Hate']))

# Check if target achieved
f1_macro = eval_results['eval_f1_macro']
if f1_macro >= 0.72:
    print(f"\n✅ SUCCESS! F1 = {f1_macro:.4f} (Target: 0.72-0.78)")
else:
    print(f"\n⚠️ F1 = {f1_macro:.4f} (Below target 0.72)")
    print("   Consider: Increase epochs, adjust learning rate, or check data quality")

## Cell 13: Save Model

In [ ]:
print("="*60)
print("💾 SAVING MODEL")
print("="*60)

save_path = "./videberta_toxic_final"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print(f"✅ Model saved to: {save_path}")
print(f"\n🎉 Done! Expected F1: 0.72-0.78")
print(f"   Actual F1: {eval_results['eval_f1_macro']:.4f}")

## Cell 14: Download Model (Optional)

In [ ]:
# Zip model folder
!zip -r videberta_toxic_final.zip videberta_toxic_final/

# Download
from google.colab import files
files.download('videberta_toxic_final.zip')

print("✅ Model downloaded!")

## Cell 15: Test Inference

In [ ]:
def predict_text(title, comment, model, tokenizer):
    """
    Predict label for title-comment pair
    """
    # Tokenize with text_pair
    inputs = tokenizer(
        text=title,
        text_pair=comment,
        truncation=True,
        max_length=Config.MAX_LENGTH,
        return_tensors="pt"
    )
    
    # Move to device
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    # Predict
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
        pred = torch.argmax(probs, dim=1)
    
    label_names = ['Clean', 'Toxic', 'Hate']
    pred_label = label_names[pred.item()]
    confidence = probs[0][pred.item()].item()
    
    print(f"\n📝 Title: {title}")
    print(f"💬 Comment: {comment}")
    print(f"\n🏷️ Prediction: {pred_label} ({confidence:.2%})")
    print(f"📊 Probabilities:")
    for i, (name, prob) in enumerate(zip(label_names, probs[0])):
        bar = "█" * int(prob.item() * 20)
        print(f"   {name}: {prob.item():.2%} {bar}")
    
    return pred_label, confidence

# Test examples
print("="*60)
print("🧪 TEST INFERENCE")
print("="*60)

test_cases = [
    ("Video hay quá", "cảm ơn bạn đã chia sẻ"),
    ("Sản phẩm này thế nào?", "đồ rác vcl"),
    ("Boy phố mới nhú", "tệ nạn xã hội tương lai đấy"),
]

for title, comment in test_cases:
    predict_text(title, comment, model, tokenizer)
    print("-" * 60)